In [ ]:
# Data file: set CMSOPEN_DATA_FILE to the absolute path of a NanoAOD ROOT file
# before starting Jupyter, e.g. the 2012C DoubleMuParked file described in DATA.md.
import os
from pathlib import Path

if not os.environ.get("CMSOPEN_DATA_FILE"):
    raise RuntimeError(
        "CMSOPEN_DATA_FILE is not set. Start Jupyter with "
        "CMSOPEN_DATA_FILE=/absolute/path/to/file.root jupyter lab ..."
    )
DATA_FILE = Path(os.environ["CMSOPEN_DATA_FILE"])
if not DATA_FILE.exists():
    raise FileNotFoundError(f"CMSOPEN_DATA_FILE points to a missing file: {DATA_FILE}")
print(f"Data file: {DATA_FILE}")

# Layer 2: muons as physics objects with `cmsopen.objects`

The detector reports each muon as transverse momentum $p_T$, pseudorapidity $\eta = -\ln\tan(\theta/2)$, azimuthal angle $\phi$ and an assigned mass $m_\mu = 0.1057$ GeV. From these follow the Cartesian components

$$p_x = p_T\cos\phi,\quad p_y = p_T\sin\phi,\quad p_z = p_T\sinh\eta,\quad E = \sqrt{p^2 + m^2},\ p = p_T\cosh\eta.$$

A `Muon` object holds *all* muons in a set of events as jagged arrays: one list per event.

In [ ]:
import awkward as ak
import numpy as np

import cmsopen
from cmsopen import Muon, open_events

try:
    import vector
except ImportError:
    vector = None

print(f"cmsopen imported from: {cmsopen.__file__}")
status = vector.__version__ if vector else "not installed"
print(f"vector library: {status} (pip install vector to enable the cross-check)")

## Load the first 100,000 events as a `Muon` collection

In [ ]:
N_EVENTS = 100_000
with open_events(DATA_FILE) as events:
    muons = Muon.from_events(events, entry_stop=N_EVENTS)
p4 = muons.p4

print(repr(muons))
print(f"events: {len(muons):,}   muons: {int(ak.sum(muons.num)):,}")
print(f"muons per event, first 5 events (.num): {muons.num[:5].tolist()}")

## Kinematics of the muons in event 0

Note how $|p_z|$ exceeds $p_T$ for $|\eta| > 0.88$ (where $\sinh\eta > 1$), and how $E \approx p$ because these muons are far from rest.

In [ ]:
print("event 0 muons (GeV, radians):")
columns = {
    "pt": p4.pt[0], "eta": p4.eta[0], "phi": p4.phi[0], "charge": muons.charge[0],
    "E": p4.energy[0], "px": p4.px[0], "py": p4.py[0], "pz": p4.pz[0],
}
print("  #  " + "".join(f"{name:>11}" for name in columns))
for j in range(int(muons.num[0])):
    cells = "".join(
        f"{int(v[j]):>11d}" if name == "charge" else f"{float(v[j]):>11.5f}" for name, v in columns.items()
    )
    print(f"  {j:<3}{cells}")

## Self-checks

1. **Mass shell.** $E^2 - |\vec p|^2 = m^2$ for every muon. $E^2$ and $|\vec p|^2$ are large and nearly equal, so their difference is exact only to a few float64 rounding units of $E^2$; the tolerance is relative to $E^2$.
2. **Known values** of the first event's muon $p_T$ (2012C file only).
3. **Independent implementation:** agreement with the `vector` library, if installed.
4. **Structure:** the four-momentum arrays have the same particle count per event as `.num`.

In [ ]:
class Checks:
    """Collects self-check results and prints one OK/FAIL/SKIP line each."""

    def __init__(self):
        self.failures = []

    def check(self, name, ok, detail):
        """Print one OK or FAIL line and remember failures."""
        print(f"{'OK' if ok else 'FAIL'}: {name} ({detail})")
        if not ok:
            self.failures.append(name)

    def skip(self, name, reason):
        """Print one SKIP line (a check that cannot run here)."""
        print(f"SKIP: {name} ({reason})")

In [ ]:
KNOWN_FILE = "45F7AE81-7A79-4CB1-998E-7BD156490294.root"
KNOWN_FIRST_PTS = [71.24144, 25.51886, 3.74997]
SHELL_TOLERANCE = 16 * np.finfo(np.float64).eps


def flat(x):
    """Flatten a jagged awkward array into a 1-D numpy array."""
    return ak.to_numpy(ak.flatten(x, axis=None))


checks = Checks()

px, py, pz, e = (flat(c) for c in p4.as_cartesian())
e2 = e**2
residual = e2 - (px**2 + py**2 + pz**2) - flat(p4.mass) ** 2
rel = np.abs(residual) / e2
checks.check(
    f"E^2 - |p|^2 = m^2 for all {len(e):,} muons",
    bool(np.all(rel <= SHELL_TOLERANCE)),
    f"max abs error {np.abs(residual).max():.3g} GeV^2, max error relative to E^2 {rel.max():.3g}, "
    f"tolerance {SHELL_TOLERANCE:.3g}",
)

if DATA_FILE.name == KNOWN_FILE:
    pts = p4.pt[0].to_numpy()
    checks.check(
        "first-event muon pts",
        len(pts) == len(KNOWN_FIRST_PTS) and np.allclose(pts, KNOWN_FIRST_PTS, rtol=0, atol=1e-5),
        f"{pts.round(5).tolist()}, expected {KNOWN_FIRST_PTS}",
    )
else:
    checks.skip("first-event muon pts", f"{DATA_FILE.name} is not the known 2012C file {KNOWN_FILE}")

if vector is None:
    checks.skip("agreement with vector library", "vector is not installed")
else:
    v = vector.zip({"pt": p4.pt, "eta": p4.eta, "phi": p4.phi, "mass": p4.mass})
    worst = 0.0
    ok = True
    for ours, theirs in [(px, v.px), (py, v.py), (pz, v.pz), (e, v.E)]:
        theirs = flat(theirs)
        ok &= bool(np.allclose(ours, theirs, rtol=1e-12, atol=1e-9))
        worst = max(worst, float(np.max(np.abs(ours - theirs) / np.maximum(np.abs(theirs), 1e-300))))
    checks.check(
        f"agreement with vector {vector.__version__} for all muons (px, py, pz, E)",
        ok,
        f"max relative difference {worst:.3g}",
    )

same = bool(ak.all(ak.num(p4.pt, axis=1) == muons.num)) and len(p4.pt) == len(muons.num)
checks.check("p4.pt counts per event equal .num", same, f"{len(muons):,} events compared")

if checks.failures:
    raise AssertionError(f"{len(checks.failures)} check(s) failed: {', '.join(checks.failures)}")
print("All checks passed")